# Validación final y diagnóstico experimental

Este notebook diagnostica resultados ya obtenidos sin reentrenar ni seleccionar con prueba. Conserva prueba como referencia histórica y usa validación para estudios ampliados. Los intervalos bootstrap remuestrean usuarios, no filas de candidatos.

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.stats import ks_2samp
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'ml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from ml.recommendation import ranking_metrics, minmax_by_user, weighted_hybrid_score
from ml.inference import RecommendationEngine
ML = ROOT / 'data' / 'ml'; CONTENT = ROOT / 'artifacts' / 'content_tfidf'; HYBRID = ROOT / 'artifacts' / 'hybrid'
TOP_K, SEED = 10, 2026
valid = pd.read_parquet(CONTENT / 'valid_candidate_scores.parquet'); test = pd.read_parquet(CONTENT / 'test_candidate_scores.parquet')
hybrid_config = json.loads((HYBRID / 'config.json').read_text()); weight = float(hybrid_config['content_weight'])
rng = np.random.default_rng(SEED)


## 1. Comparabilidad y distribución entre validación y prueba
Se verifica que cada modelo usa las mismas filas de candidatos dentro de cada partición. Después se comparan actividad, popularidad del relevante y distribución de positivos; diferencias aquí pueden explicar cambios sin atribuirlos automáticamente a sobreajuste.

In [2]:
raw_valid = pd.read_parquet(ML / 'evaluation_v2_valid.parquet'); raw_test = pd.read_parquet(ML / 'evaluation_v2_test.parquet')
for label, scored, raw in [('validación', valid, raw_valid), ('prueba', test, raw_test)]:
    assert scored[['user_idx','book_idx','relevant']].equals(raw[['user_idx','book_idx','relevant']])
    assert scored.groupby('user_idx').relevant.sum().eq(1).all()
summary = pd.DataFrame([{'partición': name, 'usuarios': frame.user_idx.nunique(), 'candidatos_por_usuario': len(frame) / frame.user_idx.nunique(), 'actividad_mediana': frame.drop_duplicates('user_idx').user_train_interactions.median(), 'popularidad_relevante_mediana': frame.drop_duplicates('user_idx').target_train_popularity.median()} for name, frame in [('validación', valid), ('prueba', test)]])
display(summary)
for column in ['user_train_interactions', 'target_train_popularity']:
    a = valid.drop_duplicates('user_idx')[column]; b = test.drop_duplicates('user_idx')[column]
    print(column, ks_2samp(a, b))
display(Markdown('**Interpretación.** La igualdad fila a fila confirma candidatos, exclusiones y relevancia comunes para los modelos dentro de cada partición. Una diferencia de distribución entre validación y prueba puede modificar el ranking de contenido, híbrido o popularidad sin implicar por sí sola sobreajuste.'))


,partición,usuarios,candidatos_por_usuario,actividad_mediana,popularidad_relevante_mediana
0,validación,5000,1001.0,10.0,416.0
1,prueba,5000,1001.0,11.0,342.0


user_train_interactions KstestResult(statistic=np.float64(0.0212), pvalue=np.float64(0.21115228503506395), statistic_location=np.int64(9), statistic_sign=np.int8(1))
target_train_popularity KstestResult(statistic=np.float64(0.036), pvalue=np.float64(0.003065315274532746), statistic_location=np.int64(820), statistic_sign=np.int8(-1))


**Interpretación.** La igualdad fila a fila confirma candidatos, exclusiones y relevancia comunes para los modelos dentro de cada partición. Una diferencia de distribución entre validación y prueba puede modificar el ranking de contenido, híbrido o popularidad sin implicar por sí sola sobreajuste.

## 2. Bootstrap y contribución híbrida
Se remuestrean usuarios con reemplazo para intervalos al 95%. La combinación se analiza comparando rangos y una fusión sin normalización, solo como diagnóstico: el peso ya quedó seleccionado en validación y no se cambia con prueba.

In [3]:
def with_hybrid(frame):
    result = minmax_by_user(frame, 'content_score', 'content_norm'); result = minmax_by_user(result, 'collaborative_score', 'collaborative_norm')
    result['hybrid_score'] = weighted_hybrid_score(result.content_norm.to_numpy(), result.collaborative_norm.to_numpy(), weight)
    result['raw_fusion_diagnostic'] = weight * result.content_score + (1 - weight) * result.collaborative_score
    return result
valid_h, test_h = with_hybrid(valid), with_hybrid(test)
def bootstrap(frame, column, n=1000):
    detail, point = ranking_metrics(frame, column, TOP_K); values = detail[['precision_at_k','recall_at_k','ndcg_at_k']].to_numpy(); draws = np.mean(values[rng.integers(0, len(values), size=(n, len(values)))], axis=1)
    return {metric: (point[metric], *np.quantile(draws[:, i], [.025, .975])) for i, metric in enumerate(point)}
rows=[]
for label, column in [('Popularidad','popularity_score'),('Colaborativo','collaborative_score'),('Contenido','content_score'),('Híbrido','hybrid_score')]:
    for metric, values in bootstrap(test_h, column).items(): rows.append({'modelo': label, 'métrica': metric, 'estimación': values[0], 'ic95_bajo': values[1], 'ic95_alto': values[2]})
display(pd.DataFrame(rows))
_, normalized_metrics = ranking_metrics(valid_h, 'hybrid_score', TOP_K); _, raw_metrics = ranking_metrics(valid_h, 'raw_fusion_diagnostic', TOP_K)
display(pd.DataFrame([{'fusión': 'min-max por usuario', **normalized_metrics}, {'fusión': 'cruda, diagnóstica', **raw_metrics}]))
print(valid_h[['content_norm','collaborative_norm','hybrid_score']].corr())


,modelo,métrica,estimación,ic95_bajo,ic95_alto
0,Popularidad,precision_at_k,0.037320,0.036059,0.038660
1,Popularidad,recall_at_k,0.373200,0.360590,0.386605
2,Popularidad,ndcg_at_k,0.245294,0.235263,0.255070
3,Colaborativo,precision_at_k,0.013720,0.012780,0.014740
4,Colaborativo,recall_at_k,0.137200,0.127800,0.147400
5,Colaborativo,ndcg_at_k,0.079582,0.073504,0.085923
6,Contenido,precision_at_k,0.031420,0.030219,0.032600
7,Contenido,recall_at_k,0.314200,0.302195,0.326005
8,Contenido,ndcg_at_k,0.209847,0.200627,0.219057
9,Híbrido,precision_at_k,0.035100,0.033779,0.036440


,fusión,precision_at_k,recall_at_k,ndcg_at_k
0,min-max por usuario,0.04324,0.4324,0.304977
1,"cruda, diagnóstica",0.03040,0.3040,0.213398


                    content_norm  collaborative_norm  hybrid_score
content_norm            1.000000            0.002905      0.890920
collaborative_norm      0.002905            1.000000      0.456748
hybrid_score            0.890920            0.456748      1.000000


## 3. Evaluación ampliada en validación
Se reutiliza la evaluación de 1,000 usuarios con 5,000 candidatos no vistos. Popularidad, colaborativo, contenido e híbrido reciben el mismo conjunto. También se mide coste aproximado de fusión y ranking; los tiempos de entrenamiento no son tiempos de inferencia.

In [4]:
broad = pd.read_parquet(CONTENT / 'valid_broad_candidate_scores.parquet'); broad_h = with_hybrid(broad)
start = time.perf_counter(); rows=[]
for label, column in [('Popularidad','popularity_score'),('Colaborativo','collaborative_score'),('Contenido','content_score'),('Híbrido','hybrid_score')]:
    _, metrics = ranking_metrics(broad_h, column, TOP_K); rows.append({'modelo': label, 'usuarios': broad_h.user_idx.nunique(), **metrics})
ranking_seconds = time.perf_counter() - start
memory_mib = broad_h.memory_usage(deep=True).sum() / 2**20
display(pd.DataFrame(rows)); print(f'Ranking/fusión: {ranking_seconds:.2f}s; memoria aproximada de candidatos+puntuaciones: {memory_mib:.1f} MiB')
display(Markdown('**Interpretación.** Al aumentar candidatos, las métricas bajan porque el ranking es más competitivo. Esta evaluación sigue siendo muestreada: la popularidad y el contenido pueden comportarse distinto frente al catálogo elegible completo.'))


,modelo,usuarios,precision_at_k,recall_at_k,ndcg_at_k
0,Popularidad,1000,0.0209,0.209,0.123957
1,Colaborativo,1000,0.0059,0.059,0.036622
2,Contenido,1000,0.0232,0.232,0.149828
3,Híbrido,1000,0.0250,0.250,0.161138


Ranking/fusión: 8.21s; memoria aproximada de candidatos+puntuaciones: 290.9 MiB


**Interpretación.** Al aumentar candidatos, las métricas bajan porque el ranking es más competitivo. Esta evaluación sigue siendo muestreada: la popularidad y el contenido pueden comportarse distinto frente al catálogo elegible completo.

## 4. Catálogo completo elegible: muestra de validación
Para una muestra controlada de 200 usuarios se puntúan todos los libros entrenables no vistos. Esto no ajusta pesos; estima la diferencia entre negativos muestreados y catálogo completo, con memoria acotada al puntuar usuario por usuario.

In [5]:
engine = RecommendationEngine(ROOT)
targets = valid_h[valid_h.relevant].set_index('user_idx').book_idx.to_dict(); sampled_users = rng.choice(np.array(sorted(targets)), size=min(200, len(targets)), replace=False)
popularity = engine.train.groupby('book_idx').size().reindex(range(len(engine.book_mapping)), fill_value=0).to_numpy()
catalog = np.arange(len(engine.book_mapping)); full_rows=[]; start=time.perf_counter()
for user in sampled_users:
    candidates = np.setdiff1d(catalog, np.fromiter(engine.known_items[int(user)], dtype=np.int64), assume_unique=False); target=int(targets[int(user)])
    collab=engine._collaborative_scores(int(user), candidates); content=engine._content_scores_for_user(int(user), candidates)
    scale=lambda x: (x-x.min())/(x.max()-x.min()) if x.max()>x.min() else np.zeros_like(x)
    hybrid=weighted_hybrid_score(scale(content), scale(collab), weight)
    for label, score in [('Popularidad', popularity[candidates]), ('Colaborativo', collab), ('Contenido', content), ('Híbrido', hybrid)]:
        rank=int(np.where(candidates[np.argsort(-score)] == target)[0][0])+1; hit=rank<=TOP_K; full_rows.append({'modelo':label,'precision_at_k':hit/TOP_K,'recall_at_k':float(hit),'ndcg_at_k':1/np.log2(rank+1) if hit else 0.0})
full_results=pd.DataFrame(full_rows).groupby('modelo', as_index=False).mean(); display(full_results); print(f'Tiempo catálogo completo: {time.perf_counter()-start:.2f}s para {len(sampled_users)} usuarios')


,modelo,precision_at_k,recall_at_k,ndcg_at_k
0,Colaborativo,0.0010,0.010,0.005655
1,Contenido,0.0070,0.070,0.035946
2,Híbrido,0.0095,0.095,0.044821
3,Popularidad,0.0030,0.030,0.009985


Tiempo catálogo completo: 21.02s para 200 usuarios


## 5. Conclusiones académicas
- La factorización redujo MSE y RMSE por épocas, pero la función MSE optimiza error de estrellas, no la posición de relevantes en Top-K.
- Popularidad es un baseline fuerte porque concentra señales frecuentes; debe conservarse como alternativa para usuarios sin perfil o cuando los modelos personalizados no demuestren ventaja.
- Contenido ofrece explicabilidad, similitud libro-libro y cold start, pero depende de cobertura y vocabulario de metadatos; no modela gustos latentes compartidos.
- El híbrido aportó complementariedad en validación, pero no superó popularidad en prueba. Por tanto, no puede declararse estrategia ganadora sin más validación.
- La poda previa al split, `date_added`, un único relevante y negativos muestreados limitan la generalización. Para integración inicial, usar popularidad como fallback y habilitar contenido/híbrido de forma observada o controlada, sin prometer superioridad global.